# PerPos Hierarchical Char Transformer — enwik8 (canonical kernel)

Provenance: imported from the user's prior Kaggle TPU kernel.
(TPU v5e-8, Keras/TF).

Local edits applied when importing into this repo:
- Scrubbed hardcoded W&B API key. Now reads `WANDB_API_KEY` from env.
- `SWEEP_ID` / `PROJECT` / `ENTITY` now strictly env-driven
  (`WANDB_SWEEP_ID` / `WANDB_PROJECT` / `WANDB_ENTITY`) with no
  hardcoded defaults that would leak personal usernames in the
  public repo.
- Added `val/bpc` and `val/best_bpc` to `WBMetricRenamer.on_epoch_end`,
  computed as `val_loss / math.log(2)`. **This is the target metric
  for the SOTA-BPC goal.**

**Goal: SOTA BPC for autoregressive char-level text generation on
enwik8 (target <= 1.05 BPC) and text8 (target <= 1.05 BPC).**
Current baseline (Sep 2025 sweep): val_loss ~2.42 -> BPC ~3.49.


In [ ]:
# Keras/TF TPU v5e-8 version of Hierarchical Char Model (PerPos AR) with online W&B sweep (single agent)
# - Strict AR per-position (no cross-position mixing)
# - Conv-only compressor, tanh → [0, K)
# - Small token-level upsampler
# - TPU-compatible tf.data, TPUStrategy, AdamW, warmup schedule
# - ALiBi disabled (coerced off in TF)
# - Includes all fixes from previous errors

# -------------------- TPU build of TensorFlow (REQUIRED on Kaggle TPU v5e-8) --------------------
# The TPU v5e-8 VM image ships the JAX TPU runtime + CPU-only `tensorflow` (no
# ConfigureDistributedTPU op). `tensorflow-tpu` is NOT preinstalled and is
# incompatible with the pre-baked JAX libtpu. Use the Kaggle-staff recipe
# (Oct 2025, herbison/tensorflow-v5e-8-oct-2025):
#   1) `uv pip uninstall --system jax`          (drop ONLY jax)
#   2) `uv pip install --system tensorflow-tpu==2.18.0 --find-links <libtpu index>`
# DO NOT pip-replace libtpu/tensorflow/jaxlib mid-container - that segfaults the
# kernel at `import tensorflow` (DeadKernelError). That is exactly what killed
# iteration 4 (log: install OK at t=105s, kernel died at t=118s). Verified to
# initialize TPU v5e-8 (8 cores) on iteration 5 / kernel v8.
import os, subprocess, sys
_pre = "export PATH=\"${HOME}/.local/bin:${PATH}\"; "
_index = "https://storage.googleapis.com/libtpu-tf-releases/index.html"
def _sh(cmd):
    r = subprocess.run(cmd, shell=True, executable="/bin/bash")
    if r.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd} (rc={r.returncode})")
    print(f"$ {cmd}", flush=True)
_uv = subprocess.run(_pre + "command -v uv", shell=True, executable="/bin/bash", capture_output=True)
if _uv.returncode == 0:
    _sh(_pre + "uv pip uninstall --system jax")
    _sh(_pre + f'uv pip install --system "tensorflow-tpu==2.18.0" --find-links {_index}')
else:
    _sh(sys.executable + " -m pip uninstall -y -q jax")
    _sh(sys.executable + f" -m pip install -q --find-links {_index} tensorflow-tpu==2.18.0")
print("tensorflow-tpu==2.18.0 installed", flush=True)

import os
import math, sys, math, time, random, zipfile, subprocess
from collections import defaultdict
import requests
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view

# -------------------- Avoid W&B agent flapping while iterating --------------------
os.environ.setdefault("WANDB_AGENT_DISABLE_FLAPPING", "true")

# -------------------- W&B setup (robust import) --------------------
def pip_install(pkg):
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
    except Exception as e:
        print(f"[WARN] Could not install {pkg}: {e}")

try:
    import wandb
except ImportError:
    pip_install("wandb>=0.16.0")
    import wandb

try:
    from wandb.integration.keras import WandbCallback
except Exception:
    WandbCallback = None

# -------------------- TF / Keras setup --------------------
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TF version:", tf.__version__)

# -------------------- W&B (MANDATORY) --------------------
# These are injected by scripts/inject_wandb.py at push time (agent-side),
# so a Kaggle kernel always has them. W&B is REQUIRED: if any var is
# missing the kernel fails loudly instead of training unlogged.
SWEEP_ID = os.environ.get("WANDB_SWEEP_ID", "")  # empty => create a new sweep
PROJECT = os.environ.get("WANDB_PROJECT", "")    # required: injected by push script
ENTITY = os.environ.get("WANDB_ENTITY", "")      # required: injected by push script
WANDB_API_KEY = os.environ.get("WANDB_API_KEY", "")  # required: injected by push script
if not (PROJECT and ENTITY and WANDB_API_KEY):
    raise RuntimeError(
        "WANDB_PROJECT, WANDB_ENTITY and WANDB_API_KEY must be set via env. "
        "They are injected by scripts/inject_wandb.py before kaggle kernels push."
    )

# The entity name can drift from the username (e.g. the personal entity may
# be '<user>-custom'). Resolve the real entity from the API so wandb.init()
# doesn't fail with 'entity not found during upsertBucket'. This overrides
# any stale WANDB_ENTITY value baked in by scripts/inject_wandb.py.
try:
    import wandb as _wandb
    _api = _wandb.Api()
    _resolved_entity = _api.default_entity
    if _resolved_entity and _resolved_entity != ENTITY:
        print(f"[W&B] overriding WANDB_ENTITY {ENTITY!r} -> {_resolved_entity!r} (resolved via API)", flush=True)
        ENTITY = _resolved_entity
        os.environ["WANDB_ENTITY"] = _resolved_entity
except Exception as _wb_err:
    print(f"[WARN] could not resolve W&B entity via API, keeping {ENTITY!r}: {_wb_err}", flush=True)

# -------------------- Precision & Strategy --------------------
MIXED_PRECISION = True  # set False if you want pure float32
if MIXED_PRECISION:
    try:
        from tensorflow.keras import mixed_precision
        mixed_precision.set_global_policy("mixed_bfloat16")
        print("Using mixed_bfloat16 policy")
    except Exception as e:
        print("[WARN] Could not enable mixed_bfloat16:", e)

def get_strategy():
    # Kaggle TPU v5e-8 (accelerator=TpuV5E8). No silent CPU fallback: the
    # kernel is configured for TPU and must fail loudly if TPU is absent.
    try:
        resolver = tf.distribute.cluster_resolver.TPUClusterResolver(tpu="local")
        tf.config.experimental_connect_to_cluster(resolver)
        tf.tpu.experimental.initialize_tpu_system(resolver)
        tpu_devices = tf.config.list_logical_devices("TPU")
        print("TPU devices:", tpu_devices, flush=True)
        if not tpu_devices:
            raise RuntimeError("No TPU devices found despite accelerator=TpuV5E8")
        return tf.distribute.TPUStrategy(resolver)
    except Exception as e:
        raise RuntimeError(
            "TPU v5e-8 required but not available. Ensure kernel-metadata.json "
            'uses "accelerator": "TpuV5E8". Root error: %r' % (e,)
        ) from e

STRATEGY = get_strategy()
print("Strategy:", type(STRATEGY).__name__, "| replicas:", STRATEGY.num_replicas_in_sync, flush=True)

# -------------------- Seed --------------------
def set_seed(seed=1337):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

# -------------------- Fixed constraints & default config --------------------
FIXED = {
    "dataset": "enwik8",
    "limit_chars": 10_000_000,
    "block_size": 512,
    "stride": 256,
    "ch": 2,  # chunk size (sweep winner true-wood-14: 2.0549 BPC)
}

CONFIG = {
    "project": PROJECT,
    "seed": 1337,
    "epochs": 40,
    "patience": 4,
    "batch_size": 384,           # TPU v5e-8: 48 seq/replica; LR auto-scaled by batch/256
    "val_frac": 0.1,
    "warmup_steps": 150,    # linear warmup steps (~4% of ~3680 total @ batch 384)
    "steps_per_execution": 16,   # TPU host/device sync amortization

    # Backbone
    "d_model": 768,
    "quantized_dim": 64,
    "quantization_levels": 256,
    "compressor_causal": True,
    "learned_pos": False,
    "compressor_emb_dim": 128,
    "bigram_buckets": 0,  # iter-16 (v24) bigram-hash REGRESSED 1.8833->1.9049 BPC; disabled (rec 62 code kept, off)
    "bigram_dim": 128,    # bigram embedding dim (== compressor_emb_dim for add)

    # Chunk-level (PerPos) decoder - AR over chunk index only
    "chunk_decoder_type": "perpos",
    "chunk_dec_layers": 8,
    "chunk_nhead": 12,
    "chunk_dec_dim_ff": 1024,
    "chunk_dec_dropout": 0.1,
    "chunk_local_window": 64,  # local-causal (sweep winner)
    "chunk_use_alibi": False,    # coerced False in TF version
    "chunk_qk_norm": True,       # LayerNorm on Q/K in chunk attention (modded-nanogpt rec 5)

    # Token-level decoder (SMALL upsampler - must stay small or it becomes a
    # second GPT transformer, undermining the hierarchy)
    "token_softcap": 15.0,  # logits = softcap*tanh(logits/softcap) (modded-nanogpt record 5/18)
    "token_emb_dim": 128,
    "token_dec_layers": 1,   # CAP 2 (sweep winner: small upsampler)
    "token_dec_nhead": 4,
    "token_dec_dim_ff": 128, # CAP 256 (sweep winner)
    "token_dec_dropout": 0.0,
    "token_local_window": 128,
    "token_window_warmup_epochs": 12,  # grow token attn window over first ~30% epochs (modded-nanogpt rec 13)
    "token_window_start": 32,
    "mtp_depth": 2,   # MTP aux heads: predict char at +2/+3 (modded-nanogpt rec 53/60/88)
    "mtp_weight": 0.25,   # REBALANCE iter-20: aux dominated train grad (train/loss 3.35 vs val 1.43); halve to favor main CE,

    # Optim
    "lr": 9e-4,  # will be scaled by batch_size/256
    "weight_decay": 0.1,
    "clip_grad": 0.5,

    # Logging/sampling
    "log_interval": 50,  # not used directly in Keras fit (per-step logging)
    "sample_every": 8,
    "sample_prompt": "Anarchism is a political philosophy advocating for the abolition of all hierarchical and coercive institutions...",
    "temperature": 1.0,
}

SWEEP_SPACE = {
    "d_model": {"values": [512, 768, 1024]},
    "chunk_dec_layers": {"values": [4, 6, 8]},   # CAP 8
    "chunk_dec_dim_ff": {"values": [256, 512, 1024, 2048]},  # CAP 2048
    "chunk_nhead": {"values": [2, 3, 4, 6, 8]},
    "chunk_local_window": {"values": [64, 128, None]},
    "chunk_use_alibi": {"values": [False, True]},  # coerced False in TF

    "token_emb_dim": {"values": [64, 128]},
    "token_dec_layers": {"values": [1, 2]},   # CAP 2 (small upsampler)
    "token_dec_nhead": {"values": [1, 2, 4]},
    "token_dec_dim_ff": {"values": [64, 128, 256]},  # CAP 256
    "token_dec_dropout": {"values": [0.0, 0.1]},

    "quantized_dim": {"values": [32, 64, 128]},
    "quantization_levels": {"values": [256]},
    "compressor_emb_dim": {"values": [32, 64, 128]},
    "learned_pos": {"values": [True, False]},

    "batch_size": {"values": [384]},  # committed budget: batch 384 (TPU v5e-8, 48 seq/replica)
    "lr": {"values": [6e-4, 9e-4, 1.2e-3]},
    "weight_decay": {"values": [0.01, 0.05, 0.1]},
    "clip_grad": {"values": [0.5, 1.0]},
    "temperature": {"values": [1.0]},
    "seed": {"values": [1337]},
}

RUN_MODE = "run"  # "run" or "sweep_online_single_agent"; verification run until TPU+W&B confirmed
TRIALS_COUNT = int(os.environ.get("WANDB_AGENT_TRIALS", "24"))

# -------------------- Data: enwik8 (Kaggle autodetect) --------------------
def _find_enwik8_in_kaggle_input():
    base = "/kaggle/input"
    if not os.path.exists(base):
        return None
    for root, dirs, files in os.walk(base):
        if "enwik8" in files:
            return os.path.join(root, "enwik8")
        if "enwik8.zip" in files:
            return os.path.join(root, "enwik8.zip")
    return None

def _ensure_enwik8_local(limit_chars=5_000_000):
    p = _find_enwik8_in_kaggle_input()
    if p:
        if p.endswith(".zip"):
            print(f"Found enwik8.zip at {p}, extracting to /kaggle/working ...")
            with zipfile.ZipFile(p, 'r') as zf:
                zf.extractall("/kaggle/working")
            return "/kaggle/working/enwik8"
        else:
            print(f"Found enwik8 at {p}")
            return p
    if os.path.exists("enwik8"):
        return "enwik8"
    if os.path.exists("enwik8.zip"):
        print("Extracting local enwik8.zip ...")
        with zipfile.ZipFile("enwik8.zip", 'r') as zf:
            zf.extractall(".")
        return "enwik8"
    url = "http://mattmahoney.net/dc/enwik8.zip"
    try:
        print("Downloading enwik8.zip (internet must be enabled) ...")
        with requests.get(url, stream=True, timeout=60) as r:
            r.raise_for_status()
            with open("enwik8.zip", "wb") as f:
                for ch in r.iter_content(8192):
                    if ch: f.write(ch)
        print("Extracting enwik8.zip ...")
        with zipfile.ZipFile("enwik8.zip", 'r') as zf:
            zf.extractall(".")
        return "enwik8"
    except Exception as e:
        print(f"[WARN] Could not download enwik8: {e}")
        return None

def load_enwik8_text(limit_chars=5_000_000):
    path = _ensure_enwik8_local(limit_chars)
    if path is None:
        print("[FALLBACK] Using Tiny Shakespeare (internet required).")
        url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
        txt = requests.get(url, timeout=30).text
        return txt[:min(limit_chars, len(txt))]
    with open(path, "rb") as f:
        data = f.read()
    return data.decode("utf-8", errors="ignore")[:limit_chars]

# -------------------- Tokenizer --------------------
class CharTokenizer:
    # byte-level tokenizer (0=<PAD>, 1..256 = raw bytes + 1)
    def __init__(self, text=None):
        self.chars = ["<PAD>"] + [bytes([i]).decode("latin-1") for i in range(256)]
        self.stoi = {ch: i for i, ch in enumerate(self.chars)}
        self.itos = {i: ch for i, ch in enumerate(self.chars)}
        self.vocab_size = len(self.chars)

    def encode(self, s: str):
        return [b + 1 for b in s.encode("utf-8")]

    def decode(self, idxs):
        b = bytes([i - 1 for i in idxs if i != 0])
        return b.decode("utf-8", errors="replace")

# -------------------- tf.data builders --------------------
def build_tf_datasets(tokenizer, text, block_size=256, batch_size=128, stride=256, val_frac=0.1):
    ids = np.array(tokenizer.encode(text), dtype=np.int32)
    split = int((1.0 - val_frac) * len(ids))
    train_ids = ids[:split]
    val_ids   = ids[split:]

    def make_windows(id_array):
        n = len(id_array)
        max_start = n - block_size - 1
        if max_start < 0:
            raise ValueError(f"data too short: len={n} <= block_size={block_size}")
        # Vectorized: build ALL sliding windows once as a contiguous int32 array,
        # then stride-slice. Avoids per-sample tf.gather rebuilt every epoch
        # (the previous per-step bottleneck at ~0.25 s/step on TPU v5e-8).
        win_view = sliding_window_view(id_array, block_size + 1)  # [n-block, block+1]
        win = np.ascontiguousarray(win_view[::stride])            # [W, block+1]
        xs = np.ascontiguousarray(win[:, :-1])
        ys = np.ascontiguousarray(win[:, 1:])
        del win
        return xs, ys

    def make_ds(id_array, is_train=True):
        xs, ys = make_windows(id_array)
        ds = tf.data.Dataset.from_tensor_slices((xs, ys))

        options = tf.data.Options()
        options.experimental_distribute.auto_shard_policy = tf.data.experimental.AutoShardPolicy.DATA
        if is_train:
            ds = ds.shuffle(8192, reshuffle_each_iteration=True)
        ds = ds.batch(batch_size, drop_remainder=True).prefetch(tf.data.AUTOTUNE).with_options(options)
        return ds

    return make_ds(train_ids, True), make_ds(val_ids, False)

# -------------------- Model components (Keras) --------------------
class ChunkCompressorEmbedding(layers.Layer):
    def __init__(self, num_embeddings, embedding_dim, chunk_size, quantized_dim, causal=True,
                 bigram_buckets=0, bigram_dim=None, **kwargs):
        super().__init__(**kwargs)
        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim
        self.chunk_size = chunk_size
        self.quantized_dim = quantized_dim
        self.causal = causal
        self.bigram_buckets = int(bigram_buckets or 0)
        self.bigram_dim = int(bigram_dim or embedding_dim)
        if causal and (quantized_dim % chunk_size != 0):
            raise ValueError(f"quantized_dim ({quantized_dim}) must be divisible by chunk_size ({chunk_size}) when causal=True")

    def build(self, input_shape):
        # Embedding
        self.embedding = self.add_weight(
            name="embedding",
            shape=(self.num_embeddings, self.embedding_dim),
            initializer=keras.initializers.RandomNormal(stddev=0.02),
            trainable=True,
        )
        # Conv kernel + bias
        self.kernel = self.add_weight(
            name="kernel",
            shape=(self.chunk_size, self.embedding_dim, self.quantized_dim),
            initializer=keras.initializers.GlorotUniform(),
            trainable=True,
        )
        self.bias = self.add_weight(
            name="bias",
            shape=(self.quantized_dim,),
            initializer="zeros",
            trainable=True,
        )

        # Mask as non-trainable weight to avoid out-of-scope issues
        if self.causal:
            ch_per_pos = self.quantized_dim // self.chunk_size
            mask_np = np.zeros((self.chunk_size, self.embedding_dim, self.quantized_dim), dtype=np.float32)
            for oc in range(self.quantized_dim):
                pos = oc // ch_per_pos
                mask_np[:pos+1, :, oc] = 1.0
        else:
            mask_np = np.ones((self.chunk_size, self.embedding_dim, self.quantized_dim), dtype=np.float32)

        if self.bigram_buckets > 0:
            self.bigram_table = self.add_weight(
                name="bigram_table",
                shape=(self.bigram_buckets, self.bigram_dim),
                initializer=keras.initializers.RandomNormal(stddev=0.02),
                trainable=True,
            )
        else:
            self.bigram_table = None

        self.weight_mask = self.add_weight(
            name="weight_mask",
            shape=(self.chunk_size, self.embedding_dim, self.quantized_dim),
            initializer=keras.initializers.Constant(mask_np),
            trainable=False,
            dtype=self.kernel.dtype,
        )
        super().build(input_shape)

    def call(self, x, bigram_key=None):
        # x: [B*n, K] int32; bigram_key: [B*n, K] int32 hash buckets (or None)
        emb0 = tf.zeros((1, self.embedding_dim), dtype=self.embedding.dtype)
        emb_rest = self.embedding[1:]
        emb_table = tf.concat([emb0, emb_rest], axis=0)

        e = tf.nn.embedding_lookup(emb_table, tf.cast(x, tf.int32))  # [B*n, K, E]
        if self.bigram_table is not None:
            bgk = tf.cast(x, tf.int32) if bigram_key is None else tf.cast(bigram_key, tf.int32)
            bg = tf.nn.embedding_lookup(self.bigram_table, bgk)  # [B*n, K, BgD]
            e = e + tf.cast(bg, e.dtype)
        filt = self.kernel * self.weight_mask
        y = tf.nn.conv1d(e, filters=filt, stride=1, padding="VALID")  # [B*n, 1, Q]
        y = tf.squeeze(y, axis=1)
        y = y + tf.cast(self.bias, y.dtype)
        return y

class InterpolatedFloatEmbedding(layers.Layer):
    def __init__(self, input_dim: int, embedding_dim: int, **kwargs):
        super().__init__(**kwargs)
        self.input_dim = int(input_dim)
        self.embedding_dim = int(embedding_dim)
        self.ln = layers.LayerNormalization(epsilon=1e-5)

    def build(self, input_shape):
        self.embedding_matrix = self.add_weight(
            name="embedding_matrix",
            shape=(self.input_dim, self.embedding_dim),
            initializer=keras.initializers.RandomNormal(stddev=0.01),
            trainable=True,
        )
        super().build(input_shape)

    def call(self, x):
        # x: [B, N, Q] float (expected in [0, input_dim-1])
        x = tf.clip_by_value(x, 0.0, float(self.input_dim - 1))
        xf = tf.floor(x)
        xc = tf.math.ceil(x)
        wh = (x - xf)[..., None]  # [B,N,Q,1]
        wl = 1.0 - wh
        xf = tf.cast(xf, tf.int32)
        xc = tf.cast(xc, tf.int32)
        ef = tf.gather(self.embedding_matrix, xf)  # [B,N,Q,E]
        ec = tf.gather(self.embedding_matrix, xc)  # [B,N,Q,E]
        out = wl * ef + wh * ec
        out = self.ln(out)
        return out  # [B,N,Q,E]

def causal_mask(L):
    i = tf.range(L)[:, None]
    j = tf.range(L)[None, :]
    return tf.cast(i >= j, tf.bool)  # [L,L] True=allow

def local_causal_mask(L, w):
    i = tf.range(L)[:, None]
    j = tf.range(L)[None, :]
    dist = i - j
    m = tf.logical_and(dist >= 0, dist < w)
    return tf.cast(m, tf.bool)

@tf.keras.utils.register_keras_serializable(package="custom")
class ReluSquared(layers.Layer):
    """ReLU^2 MLP activation (modded-nanogpt record 5): ~1-2% val-loss gain vs GELU at fixed FLOPs."""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def call(self, x):
        return tf.square(tf.nn.relu(x))


class QKNormAttention(layers.Layer):
    """Multi-head attention with LayerNorm on Q and K (modded-nanogpt rec 5).

    Stabilizes attention logits at larger context (N=128 chunk positions at
    block 512) without changing the FLOP budget. Drop-in replacement for
    layers.MultiHeadAttention with the same call signature.
    """
    def __init__(self, num_heads, key_dim, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.num_heads = int(num_heads)
        self.key_dim = int(key_dim)
        self.dropout = float(dropout)
        self._scale = float(key_dim) ** -0.5

    def build(self, input_shape):
        d = int(input_shape[-1])
        hd = self.key_dim
        self.wq = self.add_weight(name="wq", shape=(d, self.num_heads * hd),
                                  initializer=keras.initializers.GlorotUniform())
        self.wk = self.add_weight(name="wk", shape=(d, self.num_heads * hd),
                                  initializer=keras.initializers.GlorotUniform())
        self.wv = self.add_weight(name="wv", shape=(d, self.num_heads * hd),
                                  initializer=keras.initializers.GlorotUniform())
        self.wo = self.add_weight(name="wo", shape=(self.num_heads * hd, d),
                                  initializer=keras.initializers.GlorotUniform())
        self.q_norm = layers.LayerNormalization(epsilon=1e-5)
        self.k_norm = layers.LayerNormalization(epsilon=1e-5)
        self.drop = layers.Dropout(self.dropout)
        super().build(input_shape)

    def call(self, query, value, attention_mask=None, training=False):
        # query/value: [B, T, D]; attention_mask: [B, T, S] bool (True=allowed)
        B = tf.shape(query)[0]
        T = tf.shape(query)[1]
        S = tf.shape(value)[1]
        H, hd = self.num_heads, self.key_dim

        q = tf.matmul(query, self.wq)   # [B,T,H*hd]
        k = tf.matmul(value, self.wk)
        v = tf.matmul(value, self.wv)

        def heads(t):
            return tf.reshape(t, [B, -1, H, hd])
        q, k, v = heads(q), heads(k), heads(v)
        q = self.q_norm(q)
        k = self.k_norm(k)
        q = tf.transpose(q, [0, 2, 1, 3])  # [B,H,T,hd]
        k = tf.transpose(k, [0, 2, 1, 3])  # [B,H,S,hd]
        v = tf.transpose(v, [0, 2, 1, 3])  # [B,H,S,hd]

        logits = tf.matmul(q, k, transpose_b=True) * tf.cast(self._scale, q.dtype)  # [B,H,T,S]
        if attention_mask is not None:
            m = tf.cast(attention_mask, logits.dtype)
            m = m[:, None, :, :]  # [B,1,T,S] -> broadcast over heads
            logits = logits * m + (1.0 - m) * tf.cast(-1e9, logits.dtype)
        w = tf.nn.softmax(logits, axis=-1)
        w = self.drop(w, training=training)
        o = tf.matmul(w, v)  # [B,H,T,hd]
        o = tf.transpose(o, [0, 2, 1, 3])
        o = tf.reshape(o, [B, T, H * hd])
        return tf.matmul(o, self.wo)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"num_heads": self.num_heads, "key_dim": self.key_dim, "dropout": self.dropout})
        return cfg


class EncoderBlock(layers.Layer):
    def __init__(self, d_model, nhead, dim_ff, dropout=0.1, norm_first=True, qk_norm=False, **kwargs):
        super().__init__(**kwargs)
        self.norm_first = norm_first
        self.qk_norm = qk_norm
        if qk_norm:
            self.mha = QKNormAttention(num_heads=nhead, key_dim=d_model // nhead, dropout=dropout)
        else:
            self.mha = layers.MultiHeadAttention(num_heads=nhead, key_dim=d_model // nhead, dropout=dropout)
        self.drop1 = layers.Dropout(dropout)
        self.norm1 = layers.LayerNormalization(epsilon=1e-5)

        self.ffn = keras.Sequential([
            layers.Dense(dim_ff),
            ReluSquared(),
            layers.Dropout(dropout),
            layers.Dense(d_model),
        ])
        self.drop2 = layers.Dropout(dropout)
        self.norm2 = layers.LayerNormalization(epsilon=1e-5)

    def call(self, x, training=False, attn_mask=None):
        # attn_mask: [B, T, S] boolean
        if self.norm_first:
            y = self.norm1(x, training=training)
            y = self.mha(y, y, attention_mask=attn_mask, training=training)
            x = x + self.drop1(y, training=training)
            z = self.norm2(x, training=training)
            z = self.ffn(z, training=training)
            return x + self.drop2(z, training=training)
        else:
            y = self.mha(x, x, attention_mask=attn_mask, training=training)
            x = self.norm1(x + self.drop1(y, training=training), training=training)
            z = self.ffn(x, training=training)
            return self.norm2(x + self.drop2(z, training=training), training=training)

class PerPosChunkTransformerDecoder(layers.Layer):
    def __init__(self, d_model, chunk_size, nhead, num_layers, dim_feedforward=None, dropout=0.1,
                 norm_first=True, local_window=None, use_alibi=False, qk_norm=False, **kwargs):
        super().__init__(**kwargs)
        assert d_model % chunk_size == 0
        self.d_model = d_model
        self.chunk_size = chunk_size
        self.pos_dim = d_model // chunk_size
        self.local_window = local_window
        self.use_alibi = False  # Force off in TF; not implemented
        if use_alibi:
            print("[INFO] ALiBi not implemented in this TF version. Disabling.")
        if dim_feedforward is None:
            dim_feedforward = 4 * self.pos_dim
        self.blocks = [
            EncoderBlock(self.pos_dim, nhead, dim_feedforward, dropout=dropout, norm_first=norm_first, qk_norm=qk_norm)
            for _ in range(num_layers)
        ]

    def call(self, tgt, training=False):
        # tgt: [B, N, D]
        B = tf.shape(tgt)[0]
        N = tf.shape(tgt)[1]
        K = self.chunk_size
        G = self.pos_dim

        x = tf.reshape(tgt, [B, N, K, G])
        x = tf.transpose(x, [0, 2, 1, 3])     # [B,K,N,G]
        x = tf.reshape(x, [B * K, N, G])      # [B*K,N,G]

        # attention mask over chunk index; boolean True=allowed
        if self.local_window is None:
            m = causal_mask(N)
        else:
            m = local_causal_mask(N, self.local_window)
        m = tf.broadcast_to(m[None, ...], [B * K, N, N])

        for blk in self.blocks:
            x = blk(x, training=training, attn_mask=m)

        x = tf.reshape(x, [B, K, N, G])
        x = tf.transpose(x, [0, 2, 1, 3])     # [B,N,K,G]
        x = tf.reshape(x, [B, N, K * G])      # [B,N,D]
        return x

class TokenLevelDecoderCausal(layers.Layer):
    def __init__(self, d_model, chunk_size, vocab_size,
                 token_emb_dim=256, num_layers=6, nhead=4, dim_feedforward=None,
                 dropout=0.1, local_window=None, softcap=15.0,
                 mtp_depth=0, **kwargs):
        super().__init__(**kwargs)
        self.chunk_size = chunk_size
        self.vocab_size = vocab_size
        self.mtp_depth = int(mtp_depth or 0)
        self.pos_input_dim = d_model // chunk_size
        self.token_emb_dim = token_emb_dim
        self.local_window = local_window
        self.softcap = softcap
        self.window_var = tf.Variable(
            float(local_window or 1.0), trainable=False, dtype=tf.float32, name="token_window"
        )

        self.pos_projs = [layers.Dense(token_emb_dim) for _ in range(chunk_size)]
        if dim_feedforward is None:
            dim_feedforward = 4 * token_emb_dim
        self.blocks = [
            EncoderBlock(token_emb_dim, nhead, dim_feedforward, dropout=dropout, norm_first=True)
            for _ in range(num_layers)
        ]
        self.vocab_proj = layers.Dense(vocab_size)
        # Multi-token prediction (MTP) aux heads (modded-nanogpt rec 53/60/88):
        # extra heads over the same hidden states predicting the char at +2/+3
        # ahead. Aux CE is added to the training loss only (add_loss inside
        # call, gated on training=True) so val/best_loss stays the pure main
        # next-char CE and remains comparable across runs.
        self.mtp_projs = [layers.Dense(vocab_size) for _ in range(self.mtp_depth)]

    def call(self, chunk_embs, training=False):
        # chunk_embs: [B, N, D] with D = K*G
        B = tf.shape(chunk_embs)[0]
        N = tf.shape(chunk_embs)[1]
        K = self.chunk_size
        G = self.pos_input_dim

        x = tf.reshape(chunk_embs, [B, N, K, G])
        pos_embs = [self.pos_projs[p](x[:, :, p, :]) for p in range(K)]  # list of [B,N,E]
        token_embs = tf.stack(pos_embs, axis=2)                          # [B,N,K,E]
        token_embs = tf.reshape(token_embs, [B, N * K, self.token_emb_dim])  # [B,L,E]
        L = tf.shape(token_embs)[1]

        # Build attention mask (boolean)
        if self.local_window is None:
            m = causal_mask(L)
        else:
            m = local_causal_mask(L, tf.cast(self.window_var, tf.int32))
        m = tf.broadcast_to(m[None, ...], [B, L, L])

        y = token_embs
        for blk in self.blocks:
            y = blk(y, training=training, attn_mask=m)
        logits = self.vocab_proj(y)
        if self.softcap and self.softcap > 0:
            sc = tf.cast(self.softcap, logits.dtype)
            logits = sc * tf.math.tanh(logits / sc)
        aux_list = []
        if self.mtp_depth > 0:
            for k in range(self.mtp_depth):
                # predict char at offset (k+2) ahead from hidden at position t
                aux = self.mtp_projs[k](y)          # [B, L, V]
                if self.softcap and self.softcap > 0:
                    aux = sc * tf.math.tanh(aux / sc)
                aux_list.append(aux)
        return logits, aux_list  # [B, L, V], list of [B, L, V]

class HierarchicalCharTransformer(keras.Model):
    def __init__(self, vocab_size, d_model, block_size, ch,
                 quantized_dim, quantization_levels,
                 compressor_causal=True, learned_pos=False, compressor_emb_dim=64,
                 chunk_dec_layers=4, chunk_nhead=8, chunk_dec_dim_ff=None, chunk_dec_dropout=0.1,
                 chunk_local_window=None, chunk_use_alibi=False,
                 token_emb_dim=64, token_dec_layers=2, token_dec_nhead=2, token_dec_dim_ff=None,
                 token_dec_dropout=0.1, token_local_window=32, token_softcap=15.0,
                 bigram_buckets=0, bigram_dim=None, chunk_qk_norm=False,
                 mtp_depth=0, mtp_weight=0.5, **kwargs):
        super().__init__(**kwargs)
        assert block_size % ch == 0
        assert d_model % quantized_dim == 0
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.block_size = block_size
        self.ch = ch
        self.quantized_dim = quantized_dim
        self.quantization_levels = quantization_levels
        self.num_chunks = block_size // ch
        self.bigram_buckets = int(bigram_buckets or 0)
        self.mtp_depth = int(mtp_depth or 0)
        self.mtp_weight = float(mtp_weight)

        # Compressor: int -> quantized codes
        self.int_e = ChunkCompressorEmbedding(vocab_size, embedding_dim=compressor_emb_dim, chunk_size=ch,
                                              quantized_dim=quantized_dim, causal=compressor_causal,
                                              bigram_buckets=bigram_buckets, bigram_dim=bigram_dim)
        float_emb_dim = d_model // quantized_dim
        self.float_embed = InterpolatedFloatEmbedding(quantization_levels, float_emb_dim)

        # Chunk positional encoding
        self.learned_pos = learned_pos
        if learned_pos:
            self.chunk_pos_emb = self.add_weight(
                name="chunk_pos_emb",
                shape=(1, self.num_chunks, d_model),
                initializer=keras.initializers.RandomNormal(stddev=0.02),
                trainable=True,
            )
        else:
            self.chunk_sinusoids = self._build_sinusoidal_positions(self.num_chunks, d_model)

        # Chunk-level PerPos Transformer
        if chunk_dec_dim_ff is None:
            chunk_dec_dim_ff = 4 * (d_model // ch)
        self.chunk_transformer = PerPosChunkTransformerDecoder(
            d_model, ch, nhead=chunk_nhead, num_layers=chunk_dec_layers,
            dim_feedforward=chunk_dec_dim_ff, dropout=chunk_dec_dropout,
            norm_first=True, local_window=chunk_local_window, use_alibi=False,
            qk_norm=bool(chunk_qk_norm)
        )

        # Token-level upsampler
        self.token_decoder = TokenLevelDecoderCausal(
            d_model=d_model,
            chunk_size=ch,
            vocab_size=vocab_size,
            token_emb_dim=token_emb_dim,
            num_layers=token_dec_layers,
            nhead=token_dec_nhead,
            dim_feedforward=token_dec_dim_ff or 2 * token_emb_dim,
            dropout=token_dec_dropout,
            local_window=token_local_window,
            softcap=token_softcap,
            mtp_depth=int(mtp_depth or 0)
        )

    @staticmethod
    def _build_sinusoidal_positions(n, d):
        pos = np.arange(0, n)[:, None].astype(np.float32)
        div_term = np.exp(np.arange(0, d, 2).astype(np.float32) * -(math.log(10000.0) / d))
        pe = np.zeros((n, d), dtype=np.float32)
        pe[:, 0::2] = np.sin(pos * div_term)
        pe[:, 1::2] = np.cos(pos * div_term)
        return tf.constant(pe[None, ...])  # [1, n, d]

    def call(self, idx, training=False):
        # idx: [B, S] int32
        B = tf.shape(idx)[0]
        S = tf.shape(idx)[1]
        n_chunks = self.block_size // self.ch

        x_chunks = tf.reshape(idx, [B, n_chunks, self.ch])  # [B,N,K]
        x_flat = tf.reshape(x_chunks, [-1, self.ch])        # [B*N, K]
        if self.bigram_buckets > 0:
            prev = tf.concat([tf.zeros((B, 1), tf.int32), idx[:, :-1]], axis=1)  # [B,S] prev token
            key = tf.cast(prev, tf.int64) * self.vocab_size + tf.cast(idx, tf.int64)
            bg_key = tf.math.floormod(key, self.bigram_buckets)  # [B,S]
            bg_key_chunks = tf.reshape(bg_key, [B, n_chunks, self.ch])
            bg_key_flat = tf.reshape(bg_key_chunks, [-1, self.ch])  # [B*N, K]
        else:
            bg_key_flat = None
        qv = self.int_e(x_flat, bg_key_flat)                 # [B*N, Q]

        # tanh map to [0, K_levels)
        Klev = tf.cast(self.quantization_levels - 1, qv.dtype)
        scaled = (tf.math.tanh(qv) + tf.cast(1.0, qv.dtype)) * tf.cast(0.5, qv.dtype) * Klev
        scaled_chunks = tf.reshape(scaled, [B, n_chunks, self.quantized_dim])  # [B,N,Q]

        # interpolate embeddings and concat to D
        float_embs = self.float_embed(tf.cast(scaled_chunks, tf.float32))  # [B,N,Q,E]
        # cast back to compute dtype if necessary
        float_embs = tf.cast(float_embs, self.dtype)
        chunk_embs = tf.reshape(float_embs, [B, n_chunks, self.d_model])  # [B,N,D]

        # Add positions
        if self.learned_pos:
            chunk_embs = chunk_embs + tf.cast(self.chunk_pos_emb, chunk_embs.dtype)
        else:
            chunk_embs = chunk_embs + tf.cast(self.chunk_sinusoids, chunk_embs.dtype)

        # Chunk-level decoding
        trans_out = self.chunk_transformer(chunk_embs, training=training)  # [B,N,D]

        # Upsample to tokens
        logits, aux_list = self.token_decoder(trans_out, training=training)  # [B,S,V]
        if training and self.mtp_depth > 0 and aux_list:
            # MTP aux CE: at position t, aux_k[t] predicts idx[t + k + 2].
            # Targets for the kept positions are the input itself (idx).
            L = tf.shape(idx)[1]
            for k, aux in enumerate(aux_list):
                off = k + 2
                aux_pred = aux[:, :L - off]      # [B, L-off, V]
                aux_tgt = idx[:, off:]           # [B, L-off]
                ce = tf.reduce_mean(
                    tf.nn.sparse_softmax_cross_entropy_with_logits(
                        labels=aux_tgt, logits=tf.cast(aux_pred, tf.float32))
                )
                self.add_loss(self.mtp_weight * ce)
        return logits

# -------------------- Sampling --------------------
def generate(model, tokenizer, prompt, max_new_tokens, block_size, temperature, device=None):
    model.training = False
    tokens = tokenizer.encode(prompt)
    for _ in range(max_new_tokens):
        ctx = tokens[-block_size:]
        x = [0] * (block_size - len(ctx)) + ctx
        xi = tf.constant([x], dtype=tf.int32)
        logits = model(xi, training=False)  # [1, S, V]
        last = logits[:, -1, :]
        probs = tf.nn.softmax(tf.cast(last, tf.float32) / max(temperature, 1e-6), axis=-1)
        nxt = tf.random.categorical(tf.math.log(probs), num_samples=1)[0, 0].numpy().item()
        tokens.append(nxt)
    return tokenizer.decode(tokens)

# -------------------- Utils --------------------
def count_parameters(model):
    tot = int(np.sum([np.prod(v.shape) for v in model.trainable_variables]))
    print("Total trainable params:", f"{tot:,}")
    return tot

@tf.keras.utils.register_keras_serializable(package="custom")
class LinearWarmupCosineDecay(tf.keras.optimizers.schedules.LearningRateSchedule):
    """Graph-safe linear warmup then cosine decay to min_lr_ratio * base_lr.

    Unlike the iter-5 crash (LinearWarmupCosineSchedule.__call__ used a Python
    'if' on a symbolic tf.Tensor -> OperatorNotAllowedInGraphError), this
    version uses ONLY tensor ops (tf.minimum/tf.clip_by_value/tf.cos), so it
    compiles cleanly in TF graph mode on TPU.
    """

    def __init__(self, base_lr: float, warmup_steps: int, total_steps: int, min_lr_ratio: float = 0.1):
        super().__init__()
        self.base_lr = float(base_lr)
        self.warmup_steps = int(warmup_steps or 0)
        self.total_steps = int(total_steps)
        self.min_lr_ratio = float(min_lr_ratio)

    def __call__(self, step):
        step_f = tf.cast(step + 1, tf.float32)
        warm = tf.minimum(1.0, step_f / tf.cast(self.warmup_steps, tf.float32))
        t = tf.clip_by_value(
            (step_f - tf.cast(self.warmup_steps, tf.float32))
            / tf.cast(max(self.total_steps - self.warmup_steps, 1), tf.float32),
            0.0, 1.0,
        )
        cos = 0.5 * (1.0 + tf.cos(t * 3.141592653589793))
        decay = self.min_lr_ratio + (1.0 - self.min_lr_ratio) * cos
        return tf.cast(self.base_lr, tf.float32) * warm * decay

    def get_config(self):
        return {"base_lr": self.base_lr, "warmup_steps": self.warmup_steps,
                "total_steps": self.total_steps, "min_lr_ratio": self.min_lr_ratio}

    @classmethod
    def from_config(cls, config):
        return cls(**config)

class SamplingCallback(keras.callbacks.Callback):
    def __init__(self, tokenizer, cfg):
        super().__init__()
        self.tokenizer = tokenizer
        self.cfg = cfg

    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % int(self.cfg.get("sample_every", 4)) == 0:
            try:
                sample = generate(self.model, self.tokenizer, self.cfg["sample_prompt"], max_new_tokens=96,
                                  block_size=self.cfg["block_size"], temperature=self.cfg["temperature"])
                if wandb.run is not None:
                    wandb.log({"samples/text": wandb.Html("<pre>" + sample.replace("&","&amp;").replace("<","&lt;") + "</pre>")})
            except Exception as e:
                print("[WARN] sample failed:", e)

class WindowWarmupCallback(keras.callbacks.Callback):
    def __init__(self, start, target, warmup_epochs):
        super().__init__()
        self.start = int(start)
        self.target = int(target)
        self.warmup_epochs = int(warmup_epochs)

    def on_epoch_begin(self, epoch, logs=None):
        w = self.target
        if epoch < self.warmup_epochs and self.warmup_epochs > 0:
            frac = epoch / float(self.warmup_epochs)
            w = self.start + int(round((self.target - self.start) * frac))
        try:
            td = getattr(self.model, "token_decoder", None)
            if td is not None and getattr(td, "local_window", None) is not None:
                td.window_var.assign(float(w))
                if wandb.run is not None:
                    wandb.log({"schedule/token_window": int(w)})
        except Exception as e:
            print("[WARN] window warmup assign failed:", e)

class WBMetricRenamer(keras.callbacks.Callback):
    def __init__(self):
        super().__init__()
        self.best = float("inf")

    def on_epoch_end(self, epoch, logs=None):
        if logs is None or wandb.run is None:
            return
        data = {"epoch": int(epoch + 1)}
        if "loss" in logs:
            data["train/loss"] = float(logs["loss"])
        if "val_loss" in logs:
            v = float(logs["val_loss"])
            data["val/loss"] = v
            data["val/bpc"] = v / math.log(2)
            if v < self.best:
                self.best = v
                data["val/best_loss"] = self.best
                data["val/best_bpc"] = self.best / math.log(2)
        wandb.log(data)

class TimingCallback(keras.callbacks.Callback):
    """Logs per-epoch wall time and avg step time to W&B + stdout."""
    def __init__(self, steps_per_epoch):
        super().__init__()
        self.steps_per_epoch = steps_per_epoch
        self.epoch_t0 = None
        self.fit_t0 = None
        self.epoch_times = []
    def on_train_begin(self, logs=None):
        self.fit_t0 = time.time()
    def on_epoch_begin(self, epoch, logs=None):
        self.epoch_t0 = time.time()
    def on_epoch_end(self, epoch, logs=None):
        dt = time.time() - self.epoch_t0
        self.epoch_times.append(dt)
        sps = self.steps_per_epoch / dt if dt > 0 else 0.0
        if wandb.run is not None:
            wandb.log({"time/epoch_s": dt, "time/steps_per_sec": sps,
                       "time/step_s": dt / max(self.steps_per_epoch, 1)})
        print(f"[time] epoch {epoch+1}: {dt:.1f}s | {sps:.2f} steps/s | "
              f"{dt/self.steps_per_epoch:.4f} s/step", flush=True)
    def on_train_end(self, logs=None):
        total = time.time() - self.fit_t0
        if wandb.run is not None:
            wandb.log({"time/total_fit_s": total})
        print(f"[time] total fit: {total:.1f}s over {len(self.epoch_times)} epochs", flush=True)

# -------------------- Constraint-aware config coercion --------------------
def _divs(n): return [d for d in range(1, n+1) if n % d == 0]
def _nearest(x, opts): return min(opts, key=lambda v: (abs(v - x), v))

def coerce_config(raw):
    cfg = dict(raw)
    adj = []

    # Fixed items
    cfg.update(FIXED)

    # ARCHITECTURE CAPS (anti-GPT-ification guardrail, HARD cap - never coerce up):
    # The token decoder is a SMALL upsampler by design; chunk decoder is the main
    # AR model. Inflating decoder layers/FF turns PerPos into a generic GPT and
    # trivially lowers BPC without an architecture win. These caps are final.
    _CAPS = {
        "chunk_dec_layers": 8,
        "chunk_dec_dim_ff": 1024,
        "token_dec_layers": 2,
        "token_dec_dim_ff": 256,
    }
    for k, cap in _CAPS.items():
        v = cfg.get(k)
        if v is not None and (isinstance(v, (int, float)) or k.endswith("_layers")):
            try:
                iv = int(v)
            except (TypeError, ValueError):
                continue
            if iv > cap:
                old = v
                cfg[k] = cap
                adj.append(f"{k} {old}->{cfg[k]} (architecture cap {cap}, GPT-ification guard)")

    allowed_dmodel = [256, 512, 768, 1024]
    allowed_qdim = [32, 64, 128, 256]
    allowed_tokemb = [32, 64, 128, 256]
    allowed_heads = [1, 2, 3, 4, 6, 8, 12, 16, 24, 32]

    # d_model
    if cfg["d_model"] not in allowed_dmodel:
        old = cfg["d_model"]; cfg["d_model"] = _nearest(old, allowed_dmodel); adj.append(f"d_model {old}->{cfg['d_model']}")

    # quantized_dim divides d_model
    if cfg["quantized_dim"] not in allowed_qdim:
        old = cfg["quantized_dim"]; cfg["quantized_dim"] = _nearest(old, allowed_qdim); adj.append(f"quantized_dim {old}->{cfg['quantized_dim']}")
    if cfg["d_model"] % cfg["quantized_dim"] != 0:
        old = cfg["quantized_dim"]; divisors = [d for d in _divs(cfg["d_model"]) if d in allowed_qdim]
        cfg["quantized_dim"] = _nearest(old, divisors or [allowed_qdim[0]])
        if cfg["quantized_dim"] != old:
            adj.append(f"quantized_dim {old}->{cfg['quantized_dim']} to divide d_model")

    # compressor causal -> qdim % ch == 0
    if cfg.get("compressor_causal", True):
        if cfg["quantized_dim"] % cfg["ch"] != 0:
            oldq = cfg["quantized_dim"]
            valid_q = [q for q in allowed_qdim if q % cfg["ch"] == 0 and cfg["d_model"] % q == 0]
            if valid_q:
                cfg["quantized_dim"] = _nearest(oldq, valid_q); adj.append(f"quantized_dim {oldq}->{cfg['quantized_dim']} (divisible by ch)")

    # PerPos heads: target per-head dim ~32
    pos_dim = cfg["d_model"] // cfg["ch"]
    cands = [h for h in allowed_heads if h >= 1 and pos_dim % h == 0 and h <= pos_dim]
    def _dk(h): return pos_dim // h
    if cands:
        old_h = cfg.get("chunk_nhead", 4)
        best_h = min(cands, key=lambda h: (abs(_dk(h) - 32), h > old_h))
        if old_h != best_h:
            cfg["chunk_nhead"] = best_h; adj.append(f"chunk_nhead {old_h}->{cfg['chunk_nhead']} (dk~{_dk(best_h)})")

    # token_emb_dim
    if cfg["token_emb_dim"] not in allowed_tokemb:
        old = cfg["token_emb_dim"]; cfg["token_emb_dim"] = _nearest(old, allowed_tokemb); adj.append(f"token_emb_dim {old}->{cfg['token_emb_dim']}")

    # token heads
    tok_dim = cfg["token_emb_dim"]
    thcands = [h for h in allowed_heads if h >= 1 and tok_dim % h == 0 and h <= tok_dim]
    def _dk_t(h): return tok_dim // h
    if thcands:
        old_th = cfg.get("token_dec_nhead", 2)
        tgt_th = min(thcands, key=lambda h: (abs(_dk_t(h) - 32), h > old_th))
        if old_th != tgt_th:
            cfg["token_dec_nhead"] = tgt_th; adj.append(f"token_dec_nhead {old_th}->{cfg['token_dec_nhead']} (dk~{_dk_t(tgt_th)})")

    # enforce batch_size % #replicas == 0 for TPU
    replicas = STRATEGY.num_replicas_in_sync
    if replicas > 1 and (cfg["batch_size"] % replicas != 0):
        old = cfg["batch_size"]
        cfg["batch_size"] = int(math.ceil(old / replicas) * replicas)
        adj.append(f"batch_size {old}->{cfg['batch_size']} (multiple of {replicas})")

    # defaults
    cfg.setdefault("chunk_dec_layers", 4)
    cfg.setdefault("chunk_dec_dim_ff", None)
    cfg.setdefault("chunk_dec_dropout", 0.1)
    cfg.setdefault("compressor_emb_dim", 64)
    cfg.setdefault("project", PROJECT)
    cfg.setdefault("chunk_local_window", None)
    cfg.setdefault("chunk_use_alibi", False)
    cfg.setdefault("token_local_window", 32)

    # ALiBi disabled
    if cfg.get("chunk_use_alibi", False):
        cfg["chunk_use_alibi"] = False
        adj.append("chunk_use_alibi True->False (not supported in TF)")

    return cfg, ("; ".join(adj) if adj else "none")

# -------------------- Orchestration --------------------
def steps_per_epoch_estimate(cfg):
    return int(math.ceil((FIXED["limit_chars"] * (1.0 - cfg["val_frac"]) - FIXED["block_size"])
                         / (FIXED["stride"] * cfg["batch_size"])))

set_seed(CONFIG["seed"])
raw_text = load_enwik8_text(limit_chars=FIXED["limit_chars"])
print("Text len:", len(raw_text))
tokenizer_global = CharTokenizer(raw_text)

def build_model(cfg):
    with STRATEGY.scope():
        model = HierarchicalCharTransformer(
            vocab_size=257,  # PAD + 256 bytes
            d_model=cfg["d_model"],
            block_size=cfg["block_size"],
            ch=cfg["ch"],
            quantized_dim=cfg["quantized_dim"],
            quantization_levels=cfg["quantization_levels"],
            compressor_causal=cfg["compressor_causal"],
            learned_pos=cfg["learned_pos"],
            compressor_emb_dim=cfg["compressor_emb_dim"],
            chunk_dec_layers=cfg["chunk_dec_layers"],
            chunk_nhead=cfg["chunk_nhead"],
            chunk_dec_dim_ff=cfg.get("chunk_dec_dim_ff"),
            chunk_dec_dropout=cfg.get("chunk_dec_dropout", 0.1),
            chunk_local_window=cfg.get("chunk_local_window"),
            chunk_use_alibi=False,
            token_emb_dim=cfg.get("token_emb_dim", cfg["d_model"] // cfg["ch"]),
            token_dec_layers=cfg["token_dec_layers"],
            token_dec_nhead=cfg["token_dec_nhead"],
            token_dec_dim_ff=cfg.get("token_dec_dim_ff"),
            token_dec_dropout=cfg.get("token_dec_dropout", 0.1),
            token_local_window=cfg.get("token_local_window", 32),
            token_softcap=cfg.get("token_softcap", 15.0),
            bigram_buckets=cfg.get("bigram_buckets", 0),
            bigram_dim=cfg.get("bigram_dim", None),
            chunk_qk_norm=cfg.get("chunk_qk_norm", False),
            mtp_depth=cfg.get("mtp_depth", 0),
            mtp_weight=cfg.get("mtp_weight", 0.5),
        )

        # Force build so variables exist (not 0 params)
        _dummy = tf.zeros((1, cfg["block_size"]), dtype=tf.int32)
        _ = model(_dummy, training=False)

        count_parameters(model)
        base_lr = cfg["lr"] * (cfg["batch_size"] / 256.0)
        # total_steps for cosine cooldown = epochs * steps_per_epoch.
        # steps/epoch computed from the ACTUAL batch (auto-rescales when batch
        # changes, so cosine decay spans the run exactly).
        _spe = steps_per_epoch_estimate(cfg)
        lr_schedule = LinearWarmupCosineDecay(
            base_lr,
            cfg.get("warmup_steps", 0),
            total_steps=int(_spe * cfg["epochs"]),
            min_lr_ratio=cfg.get("lr_min_ratio", 0.1),
        )
        opt = keras.optimizers.AdamW(learning_rate=lr_schedule,
                                     weight_decay=cfg["weight_decay"],
                                     clipnorm=cfg["clip_grad"])
        loss = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
        model.compile(optimizer=opt, loss=loss, metrics=[],
                       steps_per_execution=cfg.get("steps_per_execution", 1))
    return model

def single_run(run_cfg):
    train_ds, val_ds = build_tf_datasets(
        tokenizer_global, raw_text,
        block_size=FIXED["block_size"],
        batch_size=run_cfg["batch_size"],
        stride=FIXED["stride"],
        val_frac=run_cfg["val_frac"],
    )
    _spe = steps_per_epoch_estimate(run_cfg)

    own_run = False
    if wandb.run is None:
        wandb.init(project=run_cfg["project"], entity=ENTITY, config=run_cfg, reinit=True)
        own_run = True

    model = build_model(run_cfg)



    callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=run_cfg["patience"], restore_best_weights=True),
    SamplingCallback(tokenizer_global, run_cfg),
    WBMetricRenamer(),  # <- logs train/loss, val/loss, val/best_loss to W&B
    TimingCallback(_spe),  # <- logs time/epoch_s, time/step_s, time/steps_per_sec
    WindowWarmupCallback(
        run_cfg.get("token_window_start", 32),
        run_cfg.get("token_local_window", 128),
        run_cfg.get("token_window_warmup_epochs", 0),
    ),
    ]
    if WandbCallback is not None:
        callbacks.append(WandbCallback(save_model=False))

    if wandb.run is not None:
        ckpt_path = os.path.join(wandb.run.dir, "best.weights.h5")
        callbacks.append(keras.callbacks.ModelCheckpoint(
            ckpt_path, monitor="val_loss", save_best_only=True, save_weights_only=True
        ))
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=run_cfg["epochs"],
        callbacks=callbacks,
        verbose=1,
    )
    best = min(history.history["val_loss"])
    if own_run:
        wandb.log({"val/best_loss": best})
        wandb.finish()
    return best

def run_online_sweep_single_agent():
    # W&B online + attach to existing sweep
    api_key = os.environ.get("WANDB_API_KEY", "")
    if not api_key:
        raise RuntimeError("WANDB_API_KEY not found. Add it as a Kaggle Secret or env var WANDB_API_KEY.")
    os.environ["WANDB_MODE"] = "online"
    wandb.login(key=api_key)

    if not SWEEP_ID:
        sweep_id = wandb.sweep(sweep_config if "sweep_config" in globals() else
                                {"method": "random",
                                 "metric": {"name": "val/loss", "goal": "minimize"},
                                 "parameters": {"lr": {"values": [1e-3, 3e-4]},
                                                "weight_decay": {"values": [0.0, 0.01]},
                                                "batch_size": {"values": [384]}}},
                               project=PROJECT, entity=ENTITY)
        print("New W&B sweep created:", sweep_id, flush=True)
    else:
        sweep_id = SWEEP_ID

    def sweep_train():
        with wandb.init(project=PROJECT, entity=ENTITY):
            base = dict(wandb.config)
            merged = {**CONFIG, **base, **FIXED}
            cfg, adj = coerce_config(merged)
            if adj != "none":
                wandb.log({"config/adjustments": adj})
                print("Config adjustments:", adj)
            set_seed(cfg["seed"])
            single_run(cfg)

    wandb.agent(SWEEP_ID, function=sweep_train, project=PROJECT, entity=ENTITY, count=TRIALS_COUNT)

# -------------------- Run mode selector --------------------
if __name__ == "__main__":
    mode = RUN_MODE = "run"  # verification run: confirm TPU + W&B before sweeping
    if mode == "run":
        merged = {**CONFIG, **FIXED}
        cfg, adj = coerce_config(merged)
        print("Config adjustments:", adj)
        best_val = single_run(cfg)
        print("Best val loss:", best_val)
    elif mode == "sweep_online_single_agent":
        run_online_sweep_single_agent()
    else:
        print(f"Unknown RUN_MODE={mode}")
